# Task 024: lenstronomy_host_decomp — Inverse Problem Tutorial

## 1. Problem Background and Mathematical Description

**Task**: Lens-source decomposition for lensed quasar host galaxy

**Paper**: lenstronomy II: A gravitational lensing software ecosystem
**Repository**: https://github.com/lenstronomy/lenstronomy

### Physical Background

X-ray CT measures attenuation line integrals. Reconstruction recovers the internal attenuation map.

### Forward Model

```
p(theta,s) = integral f(x,y) delta(x cos(theta) + y sin(theta) - s) dxdy  (Radon transform)
```

### Inverse Problem

```
Recover f(x,y) via FBP, SIRT, CGLS, or TV-regularized iterative methods
```

### Performance Metrics
- **PSNR**: 78.16 dB (model fit)
- **SSIM**: 1.0000
- **Evaluation**: Host galaxy decomposition error (gravitational lensing)

### Mathematical Formulation

Gravitational lensing bends light according to the lens equation:

$$\boldsymbol{\beta} = \boldsymbol{\theta} - \boldsymbol{\alpha}(\boldsymbol{\theta})$$

where $\boldsymbol{\beta}$ is the source position, $\boldsymbol{\theta}$ is the image position, and $\boldsymbol{\alpha}$ is the deflection angle.

**Convergence**: $\kappa(\boldsymbol{\theta}) = \frac{\Sigma(\boldsymbol{\theta})}{\Sigma_{\text{cr}}}$

**SIE lens model** deflection:
$$\alpha_1 = \frac{\theta_E}{\sqrt{1-q^2}} \arctan\left(\frac{\sqrt{1-q^2}\,\theta_1}{\psi + q^2 s}\right)$$

**Source reconstruction**: $\hat{s} = \arg\min_s \|I_{\text{obs}} - L(\boldsymbol{\theta}_{\text{lens}}) s\|^2 + \lambda \|\nabla s\|_1$


## 2. Environment Setup and Library Imports

Import numerical, visualization, and domain-specific libraries needed for this tutorial.

In [ ]:
import numpy as np
import time
from lenstronomy.Util import param_util
from lenstronomy.ImSim.image_model import ImageModel
from lenstronomy.Data.imaging_data import ImageData
from lenstronomy.Data.psf import PSF
from lenstronomy.PointSource.point_source import PointSource
from lenstronomy.LightModel.light_model import LightModel
from lenstronomy.Workflow.fitting_sequence import FittingSequence
from lenstronomy.ImSim.image_linear_solve import ImageLinearFit

## 3. Utility Functions

Helper functions providing mathematical building blocks:
`load_and_preprocess_data`

In [ ]:
def load_and_preprocess_data(numPix, deltaPix, background_rms, exp_time, fwhm, psf_type,
                              center_x, center_y, point_amp, kwargs_disk, kwargs_bulge):
    """
    Load and preprocess data: set up coordinate system, PSF, light models,
    simulate image with noise, and return all necessary components.
    
    Returns:
        dict containing:
            - data_class: ImageData object
            - psf_class: PSF object
            - lightModel: LightModel object
            - pointSource: PointSource object
            - image_sim: simulated image with noise
            - kwargs_data: data configuration
            - kwargs_psf: PSF configuration
            - kwargs_numerics: numerics configuration
            - kwargs_host: host galaxy parameters (ground truth)
            - kwargs_ps: point source parameters (ground truth)
            - light_model_list: list of light model names
            - point_source_list: list of point source model names
    """
    # Transformation matrix: pixel coordinates -> angular coordinates
    transform_pix2angle = np.array([[-deltaPix, 0], [0, deltaPix]])
    
    # Calculate the RA/Dec of the pixel (0,0) such that the image is centered at (0,0)
    cx = (numPix - 1) / 2.
    cy = (numPix - 1) / 2.
    ra_at_xy_0 = - (transform_pix2angle[0, 0] * cx + transform_pix2angle[0, 1] * cy)
    dec_at_xy_0 = - (transform_pix2angle[1, 0] * cx + transform_pix2angle[1, 1] * cy)
    
    kwargs_data = {
        'background_rms': background_rms,
        'exposure_time': exp_time,
        'ra_at_xy_0': ra_at_xy_0,
        'dec_at_xy_0': dec_at_xy_0,
        'transform_pix2angle': transform_pix2angle,
        'image_data': np.zeros((numPix, numPix))
    }
    data_class = ImageData(**kwargs_data)
    
    # Configure PSF
    kwargs_psf = {'psf_type': psf_type, 'fwhm': fwhm, 'pixel_size': deltaPix, 'truncation': 3}
    psf_class = PSF(**kwargs_psf)
    
    # Define Point Source Model
    point_source_list = ['UNLENSED']
    pointSource = PointSource(point_source_type_list=point_source_list)
    kwargs_ps = [{'ra_image': [center_x], 'dec_image': [center_y], 'point_amp': [point_amp]}]
    
    # Define Host Galaxy Model (Disk + Bulge)
    light_model_list = ['SERSIC_ELLIPSE', 'SERSIC']
    lightModel = LightModel(light_model_list=light_model_list)
    kwargs_host = [kwargs_disk, kwargs_bulge]
    
    # Numerics configuration
    kwargs_numerics = {'supersampling_factor': 1, 'supersampling_convolution': False}
    
    # Create image model for simulation
    imageModel = ImageModel(data_class, psf_class, lens_light_model_class=lightModel,
                            point_source_class=pointSource, kwargs_numerics=kwargs_numerics)
    
    # Simulate clean image
    image_sim = imageModel.image(kwargs_lens_light=kwargs_host, kwargs_ps=kwargs_ps)
    
    # Add Poisson Noise
    image_sim_counts = image_sim * exp_time
    image_sim_counts[image_sim_counts < 0] = 0
    poisson_counts = np.random.poisson(image_sim_counts)
    poisson = poisson_counts / exp_time
    poisson_noise = poisson - image_sim
    
    # Add Gaussian Background Noise
    bkg_noise = np.random.randn(*image_sim.shape) * background_rms
    
    # Combine noise
    image_sim = image_sim + bkg_noise + poisson_noise
    
    # Update Data
    kwargs_data['image_data'] = image_sim
    data_class.update_data(image_sim)
    
    return {
        'data_class': data_class,
        'psf_class': psf_class,
        'lightModel': lightModel,
        'pointSource': pointSource,
        'image_sim': image_sim,
        'kwargs_data': kwargs_data,
        'kwargs_psf': kwargs_psf,
        'kwargs_numerics': kwargs_numerics,
        'kwargs_host': kwargs_host,
        'kwargs_ps': kwargs_ps,
        'light_model_list': light_model_list,
        'point_source_list': point_source_list
    }

## 4. Forward Model

Define the forward operator mapping true model to observations.

```
p(theta,s) = integral f(x,y) delta(x cos(theta) + y sin(theta) - s) dxdy  (Radon transform)
```

Functions: `forward_operator`

In [ ]:
def forward_operator(data_class, psf_class, lightModel, pointSource, kwargs_numerics,
                     kwargs_lens_light, kwargs_ps):
    """
    Forward operator: given model parameters, compute the predicted image.
    
    Args:
        data_class: ImageData object
        psf_class: PSF object
        lightModel: LightModel object
        pointSource: PointSource object
        kwargs_numerics: numerics configuration
        kwargs_lens_light: lens light (host galaxy) parameters
        kwargs_ps: point source parameters
    
    Returns:
        y_pred: predicted image as numpy array
    """
    imageModel = ImageModel(data_class, psf_class, lens_light_model_class=lightModel,
                            point_source_class=pointSource, kwargs_numerics=kwargs_numerics)
    y_pred = imageModel.image(kwargs_lens_light=kwargs_lens_light, kwargs_ps=kwargs_ps)
    return y_pred

## 5. Inverse Solver

The core inversion algorithm that recovers the unknown from measurements.

```
Recover f(x,y) via FBP, SIRT, CGLS, or TV-regularized iterative methods
```

Functions: `run_inversion`

In [ ]:
def run_inversion(data_class, psf_class, lightModel, pointSource, kwargs_data, kwargs_psf,
                  kwargs_numerics, light_model_list, point_source_list,
                  n_particles=50, n_iterations=50):
    """
    Run the inversion/fitting process using PSO optimization.
    
    Args:
        data_class: ImageData object
        psf_class: PSF object
        lightModel: LightModel object
        pointSource: PointSource object
        kwargs_data: data configuration
        kwargs_psf: PSF configuration
        kwargs_numerics: numerics configuration
        light_model_list: list of light model names
        point_source_list: list of point source model names
        n_particles: number of PSO particles
        n_iterations: number of PSO iterations
    
    Returns:
        dict containing:
            - kwargs_result: best fit parameters
            - chain_list: fitting chain
            - fitting_time: time taken for fitting
    """
    # Define models for fitting
    kwargs_model = {
        'lens_light_model_list': light_model_list,
        'point_source_model_list': point_source_list
    }
    
    # Constraints: Joint center for all components
    kwargs_constraints = {
        'joint_lens_light_with_lens_light': [[0, 1, ['center_x', 'center_y']]],
        'joint_lens_light_with_point_source': [[0, 0], [0, 1]],
        'num_point_source_list': [1]
    }
    
    kwargs_likelihood = {'check_bounds': True, 'source_marg': False}
    
    image_band = [kwargs_data, kwargs_psf, kwargs_numerics]
    multi_band_list = [image_band]
    kwargs_data_joint = {'multi_band_list': multi_band_list, 'multi_band_type': 'multi-linear'}
    
    # Initial Parameters for Host Galaxy
    fixed_source = []
    kwargs_source_init = []
    kwargs_source_sigma = []
    kwargs_lower_source = []
    kwargs_upper_source = []
    
    # Disk (n=1 fixed)
    fixed_source.append({'n_sersic': 1})
    kwargs_source_init.append({'R_sersic': 1., 'n_sersic': 1, 'e1': 0, 'e2': 0, 'center_x': 0, 'center_y': 0})
    kwargs_source_sigma.append({'n_sersic': 0.5, 'R_sersic': 0.5, 'e1': 0.1, 'e2': 0.1, 'center_x': 0.1, 'center_y': 0.1})
    kwargs_lower_source.append({'e1': -0.5, 'e2': -0.5, 'R_sersic': 0.001, 'n_sersic': .5, 'center_x': -10, 'center_y': -10})
    kwargs_upper_source.append({'e1': 0.5, 'e2': 0.5, 'R_sersic': 10, 'n_sersic': 5., 'center_x': 10, 'center_y': 10})
    
    # Bulge (n=4 fixed)
    fixed_source.append({'n_sersic': 4})
    kwargs_source_init.append({'R_sersic': .5, 'n_sersic': 4, 'center_x': 0, 'center_y': 0})
    kwargs_source_sigma.append({'n_sersic': 0.5, 'R_sersic': 0.3, 'center_x': 0.1, 'center_y': 0.1})
    kwargs_lower_source.append({'R_sersic': 0.001, 'n_sersic': .5, 'center_x': -10, 'center_y': -10})
    kwargs_upper_source.append({'R_sersic': 10, 'n_sersic': 5., 'center_x': 10, 'center_y': 10})
    
    source_params = [kwargs_source_init, kwargs_source_sigma, fixed_source, kwargs_lower_source, kwargs_upper_source]
    
    # Point Source Parameters
    fixed_ps = [{}]
    kwargs_ps_init = [{'ra_image': [0.0], 'dec_image': [0.0]}]
    kwargs_ps_sigma = [{'ra_image': [0.01], 'dec_image': [0.01]}]
    kwargs_lower_ps = [{'ra_image': [-10], 'dec_image': [-10]}]
    kwargs_upper_ps = [{'ra_image': [10], 'dec_image': [10]}]
    
    ps_param = [kwargs_ps_init, kwargs_ps_sigma, fixed_ps, kwargs_lower_ps, kwargs_upper_ps]
    
    kwargs_params = {
        'lens_light_model': source_params,
        'point_source_model': ps_param
    }
    
    # Fitting Sequence
    fitting_seq = FittingSequence(kwargs_data_joint, kwargs_model, kwargs_constraints,
                                   kwargs_likelihood, kwargs_params)
    
    fitting_kwargs_list = [['PSO', {'sigma_scale': 1., 'n_particles': n_particles, 'n_iterations': n_iterations}]]
    
    start_time = time.time()
    chain_list = fitting_seq.fit_sequence(fitting_kwargs_list)
    kwargs_result = fitting_seq.best_fit()
    end_time = time.time()
    fitting_time = end_time - start_time
    
    return {
        'kwargs_result': kwargs_result,
        'chain_list': chain_list,
        'fitting_time': fitting_time
    }

## 6. Visualization and Metrics

Functions for evaluating and visualizing results.

Functions: `evaluate_results`

In [ ]:
def evaluate_results(data_class, psf_class, lightModel, pointSource, kwargs_numerics,
                     kwargs_result, image_sim):
    """
    Evaluate the fitting results by computing the reconstructed image and comparing
    with the observed data.
    
    Args:
        data_class: ImageData object
        psf_class: PSF object
        lightModel: LightModel object
        pointSource: PointSource object
        kwargs_numerics: numerics configuration
        kwargs_result: best fit parameters from inversion
        image_sim: observed/simulated image
    
    Returns:
        dict containing:
            - image_reconstructed: reconstructed image
            - residual: difference between observed and reconstructed
            - reconstructed_sum: sum of reconstructed image
            - true_sum: sum of observed image
            - residual_rms: RMS of residuals
    """
    imageLinearFit = ImageLinearFit(
        data_class=data_class,
        psf_class=psf_class,
        lens_light_model_class=lightModel,
        point_source_class=pointSource,
        kwargs_numerics=kwargs_numerics
    )
    
    image_reconstructed, _, _, _ = imageLinearFit.image_linear_solve(
        kwargs_lens_light=kwargs_result['kwargs_lens_light'],
        kwargs_ps=kwargs_result['kwargs_ps']
    )
    
    residual = image_sim - image_reconstructed
    reconstructed_sum = np.sum(image_reconstructed)
    true_sum = np.sum(image_sim)
    residual_rms = np.sqrt(np.mean(residual**2))
    
    return {
        'image_reconstructed': image_reconstructed,
        'residual': residual,
        'reconstructed_sum': reconstructed_sum,
        'true_sum': true_sum,
        'residual_rms': residual_rms
    }

## 7. Execution Pipeline

Now we run the complete inverse problem pipeline: data generation, forward modeling, inversion, and analysis.

### Configuration and Parameters

Set up experiment parameters: grid sizes, noise levels, regularization weights, algorithm settings.

In [ ]:
# Set random seed for reproducibility
np.random.seed(42)

# Data specifics
background_rms = 0.1
exp_time = 100.
numPix = 80
deltaPix = 0.05
fwhm = 0.1
psf_type = 'GAUSSIAN'

# Model parameters
center_x = 0.02
center_y = 0.01
point_amp = 10000

# Host galaxy parameters
e1, e2 = param_util.phi_q2_ellipticity(phi=0.3, q=0.6)
kwargs_disk = {
    'amp': 400, 'n_sersic': 1, 'R_sersic': 0.7,
    'e1': e1, 'e2': e2, 'center_x': center_x, 'center_y': center_y
}
kwargs_bulge = {
    'amp': 400, 'n_sersic': 4, 'R_sersic': 0.3,
    'center_x': center_x, 'center_y': center_y
}

### Step 1: Load and preprocess data

Intermediate processing step in the pipeline.

In [ ]:
# Step 1: Load and preprocess data
print("Loading and preprocessing data...")
data = load_and_preprocess_data(
    numPix=numPix,
    deltaPix=deltaPix,
    background_rms=background_rms,
    exp_time=exp_time,
    fwhm=fwhm,
    psf_type=psf_type,
    center_x=center_x,
    center_y=center_y,
    point_amp=point_amp,
    kwargs_disk=kwargs_disk,
    kwargs_bulge=kwargs_bulge
)
print("Data generation complete (Quasar + Host).")

### Step 2: Demonstrate forward operator

Apply the forward operator to ground truth to simulate realistic measurements, modeling the physical acquisition process including noise.

In [ ]:
# Step 2: Demonstrate forward operator
print("\nTesting forward operator...")
y_pred = forward_operator(
    data_class=data['data_class'],
    psf_class=data['psf_class'],
    lightModel=data['lightModel'],
    pointSource=data['pointSource'],
    kwargs_numerics=data['kwargs_numerics'],
    kwargs_lens_light=data['kwargs_host'],
    kwargs_ps=data['kwargs_ps']
)
print(f"Forward operator output shape: {y_pred.shape}")
print(f"Forward operator output sum: {np.sum(y_pred):.2f}")

### Step 3: Run inversion

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
# Step 3: Run inversion
print("\nStarting fitting sequence...")
inversion_result = run_inversion(
    data_class=data['data_class'],
    psf_class=data['psf_class'],
    lightModel=data['lightModel'],
    pointSource=data['pointSource'],
    kwargs_data=data['kwargs_data'],
    kwargs_psf=data['kwargs_psf'],
    kwargs_numerics=data['kwargs_numerics'],
    light_model_list=data['light_model_list'],
    point_source_list=data['point_source_list'],
    n_particles=50,
    n_iterations=50
)
print(f"Fitting completed in {inversion_result['fitting_time']:.2f} seconds.")
print("\nBest fit result:")
print(inversion_result['kwargs_result'])

### Step 4: Evaluate results

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
# Step 4: Evaluate results
print("\nEvaluating results...")
evaluation = evaluate_results(
    data_class=data['data_class'],
    psf_class=data['psf_class'],
    lightModel=data['lightModel'],
    pointSource=data['pointSource'],
    kwargs_numerics=data['kwargs_numerics'],
    kwargs_result=inversion_result['kwargs_result'],
    image_sim=data['image_sim']
)
print(f"Reconstructed Image Sum: {evaluation['reconstructed_sum']:.2f}")
print(f"True Image Sum: {evaluation['true_sum']:.2f}")
print(f"Residual RMS: {evaluation['residual_rms']:.4f}")

print("\nOPTIMIZATION_FINISHED_SUCCESSFULLY")

### Convergence Analysis

Tracking algorithm convergence is critical for understanding behavior. We plot:
- **Objective/loss** vs iteration number
- **Relative change** $\|\mathbf{x}^{(k+1)} - \mathbf{x}^{(k)}\| / \|\mathbf{x}^{(k)}\|$ to verify convergence
- **Reconstruction quality** (PSNR/SSIM) vs iteration if ground truth is available

A well-behaved algorithm should show monotonic decrease in the objective and eventual flattening.

In [ ]:
# === Convergence Analysis ===
# If the reconstruction code tracked iteration history, plot it here.
# Otherwise, this cell provides a template for convergence visualization.
import matplotlib.pyplot as plt
import numpy as np

# Example: if 'loss_history' or similar variable exists from reconstruction
try:
    # Try common variable names for convergence data
    conv_data = None
    for name in ['loss_history', 'losses', 'cost_history', 'residuals',
                  'objective', 'convergence', 'hist', 'history']:
        if name in dir():
            conv_data = eval(name)
            break
    
    if conv_data is not None and len(conv_data) > 1:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        iterations = np.arange(1, len(conv_data) + 1)
        
        axes[0].semilogy(iterations, conv_data, 'b-', linewidth=1.5)
        axes[0].set_xlabel('Iteration', fontsize=12)
        axes[0].set_ylabel('Objective Value', fontsize=12)
        axes[0].set_title('Convergence Curve', fontsize=13)
        axes[0].grid(True, alpha=0.3)
        
        if len(conv_data) > 2:
            rel_change = np.abs(np.diff(conv_data)) / (np.abs(conv_data[:-1]) + 1e-12)
            axes[1].semilogy(iterations[1:], rel_change, 'r-', linewidth=1.5)
            axes[1].set_xlabel('Iteration', fontsize=12)
            axes[1].set_ylabel('Relative Change', fontsize=12)
            axes[1].set_title('Convergence Rate', fontsize=13)
            axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        print(f"Final objective: {conv_data[-1]:.6e}")
        print(f"Total iterations: {len(conv_data)}")
    else:
        print("No convergence history found. The reconstruction may use a direct (non-iterative) method.")
except Exception as e:
    print(f"Convergence analysis skipped: {e}")
    print("Tip: Modify the reconstruction code to record loss at each iteration.")

### Error Map and Reconstruction Quality

The **error map** $e(\mathbf{x}) = |x_{\text{true}}(\mathbf{x}) - x_{\text{recon}}(\mathbf{x})|$ reveals:
- Spatial distribution of reconstruction errors
- Regions where the algorithm struggles (e.g., edges, fine details)
- Systematic biases in the reconstruction

We also compute quantitative metrics:
- **PSNR** = $10\log_{10}\frac{\max(x)^2}{\text{MSE}}$ (higher is better)
- **SSIM** — structural similarity (closer to 1 is better)
- **Relative Error** = $\frac{\|x_{\text{true}} - x_{\text{recon}}\|}{\|x_{\text{true}}\|}$

In [ ]:
# === Error Map Visualization ===
import matplotlib.pyplot as plt
import numpy as np

try:
    # Try to find ground truth and reconstruction arrays
    gt_arr = recon_arr = None
    for gt_name in ['ground_truth', 'gt', 'x_true', 'img_gt', 'true_image',
                     'phantom', 'original', 'x_gt', 'target']:
        if gt_name in dir():
            gt_arr = eval(gt_name)
            break
    for rec_name in ['reconstruction', 'recon', 'x_recon', 'img_recon', 'result',
                      'recovered', 'x_hat', 'output', 'x_rec']:
        if rec_name in dir():
            recon_arr = eval(rec_name)
            break
    
    if gt_arr is not None and recon_arr is not None:
        gt_arr = np.array(gt_arr, dtype=float)
        recon_arr = np.array(recon_arr, dtype=float)
        
        # Handle shape mismatch
        if gt_arr.shape != recon_arr.shape:
            min_shape = tuple(min(a, b) for a, b in zip(gt_arr.shape, recon_arr.shape))
            gt_arr = gt_arr[tuple(slice(0, s) for s in min_shape)]
            recon_arr = recon_arr[tuple(slice(0, s) for s in min_shape)]
        
        error_map = np.abs(gt_arr - recon_arr)
        
        if gt_arr.ndim == 2:  # 2D images
            fig, axes = plt.subplots(1, 4, figsize=(18, 4))
            
            im0 = axes[0].imshow(gt_arr, cmap='viridis')
            axes[0].set_title('Ground Truth', fontsize=12)
            plt.colorbar(im0, ax=axes[0], fraction=0.046)
            
            im1 = axes[1].imshow(recon_arr, cmap='viridis')
            axes[1].set_title('Reconstruction', fontsize=12)
            plt.colorbar(im1, ax=axes[1], fraction=0.046)
            
            im2 = axes[2].imshow(error_map, cmap='hot')
            axes[2].set_title('|Error Map|', fontsize=12)
            plt.colorbar(im2, ax=axes[2], fraction=0.046)
            
            # Histogram of errors
            axes[3].hist(error_map.ravel(), bins=50, color='steelblue', edgecolor='black', alpha=0.7)
            axes[3].set_xlabel('Absolute Error')
            axes[3].set_ylabel('Count')
            axes[3].set_title('Error Distribution', fontsize=12)
            
            for ax in axes[:3]:
                ax.axis('off')
        elif gt_arr.ndim == 1:  # 1D signals
            fig, axes = plt.subplots(2, 1, figsize=(12, 8))
            axes[0].plot(gt_arr, 'b-', label='Ground Truth', linewidth=1.5)
            axes[0].plot(recon_arr, 'r--', label='Reconstruction', linewidth=1.5)
            axes[0].legend(fontsize=11)
            axes[0].set_title('Signal Comparison', fontsize=13)
            axes[0].grid(True, alpha=0.3)
            
            axes[1].plot(error_map, 'k-', linewidth=1)
            axes[1].fill_between(range(len(error_map)), error_map, alpha=0.3, color='red')
            axes[1].set_title('Absolute Error', fontsize=13)
            axes[1].set_xlabel('Index')
            axes[1].grid(True, alpha=0.3)
        else:
            print(f"Data is {gt_arr.ndim}D — showing central slice")
            mid = gt_arr.shape[0] // 2
            fig, axes = plt.subplots(1, 3, figsize=(14, 4))
            axes[0].imshow(gt_arr[mid], cmap='viridis'); axes[0].set_title('GT (mid-slice)')
            axes[1].imshow(recon_arr[mid], cmap='viridis'); axes[1].set_title('Recon (mid-slice)')
            axes[2].imshow(error_map[mid], cmap='hot'); axes[2].set_title('Error (mid-slice)')
            for ax in axes: ax.axis('off')
        
        plt.tight_layout()
        plt.show()
        
        # Quantitative metrics
        mse = np.mean(error_map**2)
        rel_err = np.linalg.norm(error_map) / (np.linalg.norm(gt_arr) + 1e-12)
        data_range = gt_arr.max() - gt_arr.min()
        psnr = 10 * np.log10(data_range**2 / (mse + 1e-12)) if mse > 0 else float('inf')
        print(f"MSE:            {mse:.6e}")
        print(f"PSNR:           {psnr:.2f} dB")
        print(f"Relative Error: {rel_err:.6f}")
        print(f"Max Error:      {error_map.max():.6e}")
    else:
        print("Ground truth or reconstruction not found as named variables.")
        print("Modify this cell to reference your actual variable names.")
except Exception as e:
    print(f"Error map visualization skipped: {e}")

### Sensitivity Analysis

We analyze how reconstruction quality depends on key parameters:
- **Noise level**: How robust is the algorithm to increasing noise?
- **Regularization parameter** $\lambda$: What is the optimal trade-off?
- **Algorithm-specific parameters**: iterations, step size, etc.

This helps understand the algorithm's operating range and tune hyperparameters.

> **Note**: This analysis uses the variables defined earlier. If the reconstruction function
> is not available as a callable, this cell provides a template for manual parameter sweeps.

In [ ]:
# === Sensitivity / Parameter Sweep Analysis ===
import matplotlib.pyplot as plt
import numpy as np

print("Sensitivity Analysis Template")
print("=" * 50)
print()
print("To perform a full sensitivity analysis, uncomment and adapt the code below")
print("to match your specific reconstruction function and parameters.")
print()

# Template for noise sensitivity analysis:
# noise_levels = [0.01, 0.02, 0.05, 0.1, 0.2]
# psnr_values = []
# for sigma in noise_levels:
#     noisy_data = clean_data + sigma * np.random.randn(*clean_data.shape)
#     recon = run_reconstruction(noisy_data, ...)
#     psnr = compute_psnr(ground_truth, recon)
#     psnr_values.append(psnr)
#
# plt.figure(figsize=(8, 5))
# plt.plot(noise_levels, psnr_values, 'bo-', linewidth=2, markersize=8)
# plt.xlabel('Noise Level (sigma)', fontsize=12)
# plt.ylabel('PSNR (dB)', fontsize=12)
# plt.title('Reconstruction Quality vs Noise Level', fontsize=13)
# plt.grid(True, alpha=0.3)
# plt.show()

# Template for regularization parameter sweep:
# lambdas = np.logspace(-4, 1, 20)
# psnr_values = []
# for lam in lambdas:
#     recon = run_reconstruction(data, lambda_reg=lam, ...)
#     psnr = compute_psnr(ground_truth, recon)
#     psnr_values.append(psnr)
#
# plt.figure(figsize=(8, 5))
# plt.semilogx(lambdas, psnr_values, 'rs-', linewidth=2, markersize=8)
# plt.xlabel('Regularization Parameter (lambda)', fontsize=12)
# plt.ylabel('PSNR (dB)', fontsize=12)
# plt.title('L-curve: PSNR vs Regularization Strength', fontsize=13)
# plt.grid(True, alpha=0.3)
# plt.show()

print("Adapt the templates above to your specific forward model and reconstruction algorithm.")

## 8. Conclusion and Summary

This tutorial demonstrated the complete inverse problem pipeline for **lenstronomy_host_decomp**:

1. **Problem Setup**: X-ray CT measures attenuation line integrals. Reconstruction recovers the internal attenuation map.
2. **Forward Model**: Simulated measurements using the physical forward operator
3. **Inversion**: Applied the reconstruction algorithm to recover the unknown
4. **Evaluation**: Assessed reconstruction quality (PSNR=78.16 dB (model fit), SSIM=1.0000)

### Key Takeaways
- The forward model maps unknown quantities to observable measurements
- Regularization is essential to stabilize the ill-posed inverse problem
- Quantitative metrics (PSNR, SSIM) and visual comparison validate the reconstruction

### Further Reading
- Paper: lenstronomy II: A gravitational lensing software ecosystem
- Repository: https://github.com/lenstronomy/lenstronomy
